# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

Runs **FastAPI backend + Qwen 3.5 4B natively** in Google Colab.
The model is initialized automatically when the backend starts.

In [ ]:
# CELL 1: Clean and uninstall existing
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Environment cleaned")

In [ ]:
# CELL 2: Install Libraries + Qwen weights cache
!pip install fastapi==0.124.1 uvicorn==0.34.0 starlette==0.49.1 pydantic==2.12.0 pydantic-settings httpx==0.28.1 python-multipart==0.0.18 neo4j==5.23.0 pyngrok PyMuPDF==1.24.0 pdfplumber==0.11.0 transformers==4.45.0 accelerate==0.34.0 torch
print("✅ Core Libraries Installed")

In [ ]:
# CELL 3: Mount Google Drive & Set Project Root
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"✅ Working in {os.getcwd()}")

In [ ]:
# CELL 4: Validate components
import fastapi, uvicorn, transformers, torch
print(f"fastapi: {fastapi.__version__}")
print(f"GPU: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0)}")

In [ ]:
# CELL 5: Set Credentials
import os
os.environ['NEO4J_URI'] = 'neo4j+s://xxxxxxxx.databases.neo4j.io'  # ← REPLACE
os.environ['NEO4J_USER'] = 'neo4j'                                  # ← REPLACE
os.environ['NEO4J_PASSWORD'] = 'your-password-here'                  # ← REPLACE
os.environ['ENABLE_LOCAL_QWEN'] = 'true'
print("✅ Credentials set")

In [ ]:
# CELL 6: Start ngrok tunnel
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← REPLACE!
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("=" * 60)
print(f"🌐 PUBLIC URL: {public_url}")
print("=" * 60)

In [ ]:
# CELL 7: Start server with nohup
import os
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
!pkill -f uvicorn 2>/dev/null
# The backend will automatically download and start Qwen 3.5 4B upon startup!
!nohup bash -c "cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}:$PYTHONPATH uvicorn backend.main:app --host 0.0.0.0 --port 8000" > {PROJECT_ROOT}/backend.log 2>&1 &
print("🚀 Native Backend Starting. Loading Qwen takes about 1-2 minutes.\nWatch backend.log for progress.")

In [ ]:
# Utility: Watch logs (Run this cell to see Qwen loading progress)
!tail -n 25 /content/drive/MyDrive/Knowledge_graph/backend.log

In [ ]:
# CELL 8: Health check
import requests
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f"✅ Server UP: {r.json()}")
except Exception as e:
    print(f"❌ Not responding yet (Qwen might still be loading): {e}")